In [2]:
import numpy as np
import pandas as pd

In [ ]:

class MyKNNClf():

    def __init__(self, k = 3, metric = 'euclidian', weight = 'uniform'):
        self.k = k
        self.train_size = (0,0)
        self.metric = getattr(self, metric)
        self.weight = getattr(self, weight)

    def fit(self, X, y):
        self.X = X.values.astype(float)
        self.y = y.values
        self.train_size = (self.X).shape

    def predict(self, X_test):
        X_test = X_test.values.astype(float) if hasattr(X_test, 'values') else np.array(X_test).astype(float)
        pred_vector = np.zeros(X_test.shape[0])
        for idx, test_row in enumerate(X_test):
            range_vector = np.zeros(self.X.shape[0])
            for i in range(self.X.shape[0]):
                range_vector[i] = self.metric(test_row, self.X[i])
            indicies = np.argsort(range_vector)[:self.k]
            if self.weight(indicies, self.y, range_vector) < 0.5:
                pred_vector[idx] = 0
            else:
                pred_vector[idx] = 1
        return pred_vector.astype(int)

    def predict_proba(self, X_test):
        X_test = X_test.values.astype(float) if hasattr(X_test, 'values') else np.array(X_test).astype(float)
        pred_vector = np.zeros(X_test.shape[0])
        for idx, test_row in enumerate(X_test):
            range_vector = np.zeros(self.X.shape[0])
            for i in range(self.X.shape[0]):
                range_vector[i] = self.metric(test_row, self.X[i])
            indicies = np.argsort(range_vector)[:self.k]
            pred_vector[idx] = self.weight(indicies, self.y, range_vector)

        return pred_vector
    
    def euclidian(self, vec1, vec2):
        return np.sqrt(np.sum((vec1 - vec2) ** 2))

    def chebyshev(self, vec1, vec2):
        return np.max(np.abs(vec1 - vec2))

    def manhattan(self, vec1, vec2):
        return np.sum(np.abs(vec1 - vec2))

    def cosine(self, vec1, vec2):
        return 1 - (vec2.dot(vec1))/(np.sqrt(np.sum(vec1 ** 2)) * np.sqrt(np.sum(vec2 ** 2)))
    
    def uniform(self, indicies, y_class_vector, range_vector):
        return np.mean(y_class_vector[indicies])

    def rank(self, indicies, y_class_vector, range_vector):
        prob_1 = 0
        for i in range(len(indicies)):
            prob_1 += y_class_vector[indicies[i]] / (i + 1)
        deviator = np.sum(1 / np.arange(1, len(indicies) + 1))
        return prob_1 / deviator

    def distance(self, indicies, y_class_vector, range_vector):
        prob_1 = 0
        for i in range(len(indicies)):
            prob_1 += y_class_vector[indicies[i]] / range_vector[indicies[i]]
        deviator = np.sum(1 / range_vector[indicies])
        return prob_1 / deviator

                
    def __str__(self):
        return f'MyKNNClf class: k={self.k}'
